In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb
from lifelines.utils import concordance_index

ROOT = Path("data")
RAW_ROOT = ROOT / "raw"
FUZZY_DIR = ROOT / "fuzzy"
BASE_DIR = ROOT / "models" / "baseline_pinned"
AUG_DIR  = ROOT / "models" / "augmented"
INT_DIR  = ROOT / "models" / "interactions"
INT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_END = pd.Timestamp("2024-04-01").timestamp()
HOLD_END  = pd.Timestamp("2024-07-02").timestamp()

with open(ROOT / "processed" / "task_registry.json") as f:
    TASKS = json.load(f)
with open(ROOT / "processed" / "feature_list.json") as f:
    raw_FL = json.load(f)
FEATURES = raw_FL["features"]

FUZZY_COLS = ["mu_Safe", "mu_Watch", "mu_Danger", "mu_Critical", "interval_width"]
INTERACTION_BASE = ["amountUSD", "userLiquidationCount",
                    "marketLiquidationCount", "userBorrowCount"]
INTERACTION_WITH = ["mu_Critical", "mu_Danger", "interval_width"]

PARAMS = {
    "objective": "survival:cox",
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 10,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "verbosity": 0,
}
NUM_ROUNDS = 500

missing_base = [c for c in INTERACTION_BASE if c not in FEATURES]
print(f"tasks: {len(TASKS)}  base features: {len(FEATURES)}  fuzzy: {len(FUZZY_COLS)}")
print(f"interaction base cols: {INTERACTION_BASE}")
print(f"missing from FEATURES: {missing_base}")

tasks: 16  base features: 80  fuzzy: 5
interaction base cols: ['amountUSD', 'userLiquidationCount', 'marketLiquidationCount', 'userBorrowCount']
missing from FEATURES: []


In [9]:
def build_interactions(df, fz):
    df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)
    fz = fz.sort_values("timestamp", kind="stable").reset_index(drop=True)
    if len(df) != len(fz):
        raise ValueError(f"length mismatch: df={len(df)}, fz={len(fz)}  "
                         f"(nb3 memberships don't align 1:1 with raw rows)")
    for col in FUZZY_COLS:
        df[col] = fz[col].values
    for b in INTERACTION_BASE:
        if b == "amountUSD":
            b_vals = np.log1p(df[b].fillna(0))
            b_name = "amountUSD_log"
        else:
            b_vals = df[b].fillna(0)
            b_name = b
        for w in INTERACTION_WITH:
            df[f"{b_name}_x_{w}"] = b_vals * df[w].fillna(0)
    return df

INTERACTION_COLS = []
for b in INTERACTION_BASE:
    b_name = "amountUSD_log" if b == "amountUSD" else b
    for w in INTERACTION_WITH:
        INTERACTION_COLS.append(f"{b_name}_x_{w}")

FULL_FEATURES = FEATURES + FUZZY_COLS + INTERACTION_COLS
print(f"full feature set: {len(FULL_FEATURES)} columns")
print(f"interaction columns: {INTERACTION_COLS}")

full feature set: 97 columns
interaction columns: ['amountUSD_log_x_mu_Critical', 'amountUSD_log_x_mu_Danger', 'amountUSD_log_x_interval_width', 'userLiquidationCount_x_mu_Critical', 'userLiquidationCount_x_mu_Danger', 'userLiquidationCount_x_interval_width', 'marketLiquidationCount_x_mu_Critical', 'marketLiquidationCount_x_mu_Danger', 'marketLiquidationCount_x_interval_width', 'userBorrowCount_x_mu_Critical', 'userBorrowCount_x_mu_Danger', 'userBorrowCount_x_interval_width']


In [11]:
def xgb_cox_label(y_time, y_status):
    y = y_time.astype(float).copy()
    y[y_status == 0] = -y[y_status == 0]
    y[y == 0] = 1e-6
    return y

In [12]:
tid = "Borrow_Deposit"
train_path = RAW_ROOT / TASKS[0]["train_path"].replace("\\", "/")
df0 = pd.read_csv(train_path)
fz0 = pd.read_parquet(FUZZY_DIR / f"{tid}_memberships.parquet")
print(f"raw df rows: {len(df0)}, memberships rows: {len(fz0)}")

raw df rows: 882736, memberships rows: 882736


In [14]:
import time, gc
from sklearn.preprocessing import LabelEncoder

CATEGORICAL = [c for c in ["userCoinTypeMode", "userReserveMode"] if c in FEATURES]

HOLDOUT_START = pd.Timestamp("2024-04-01").timestamp()
HOLDOUT_END   = pd.Timestamp("2024-07-02").timestamp()

PARAMS = {
    "objective": "survival:cox",
    "eval_metric": "cox-nloglik",
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 10,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "verbosity": 0,
}

def load_task_with_fuzzy_and_interactions(task):
    tid = task["task_id"]
    cols = FEATURES + CATEGORICAL + ["timeDiff", "status", "timestamp"]
    cols = list(dict.fromkeys(cols))
    df = pd.read_csv(RAW_ROOT / task["train_path"].replace("\\", "/"), usecols=cols)
    for c in CATEGORICAL:
        le = LabelEncoder().fit(df[c].astype(str))
        df[c] = le.transform(df[c].astype(str))
    df[FEATURES] = df[FEATURES].fillna(0)

    fuzzy = pd.read_parquet(FUZZY_DIR / f"{tid}_memberships.parquet")
    assert len(fuzzy) == len(df), f"length mismatch {len(fuzzy)} vs {len(df)}"
    for col in FUZZY_COLS:
        df[col] = fuzzy[col].values

    # interactions
    for b in INTERACTION_BASE:
        if b == "amountUSD":
            b_vals = np.log1p(df[b].fillna(0))
            b_name = "amountUSD_log"
        else:
            b_vals = df[b].fillna(0)
            b_name = b
        for w in INTERACTION_WITH:
            df[f"{b_name}_x_{w}"] = b_vals * df[w].fillna(0)

    fit     = df[df["timestamp"] <  HOLDOUT_START].reset_index(drop=True)
    holdout = df[(df["timestamp"] >= HOLDOUT_START) & (df["timestamp"] <= HOLDOUT_END)].reset_index(drop=True)
    return fit, holdout

def fit_and_eval(fit, holdout, features):
    fit = fit.sort_values("timestamp").reset_index(drop=True)
    n = len(fit)
    n_val = int(n * 0.1)
    tr = fit.iloc[:n - n_val]
    vl = fit.iloc[n - n_val:]
    y_tr = xgb_cox_label(tr["timeDiff"].values, tr["status"].values)
    y_vl = xgb_cox_label(vl["timeDiff"].values, vl["status"].values)
    dtr = xgb.DMatrix(tr[features].values, label=y_tr)
    dvl = xgb.DMatrix(vl[features].values, label=y_vl)
    dho = xgb.DMatrix(holdout[features].values)
    bst = xgb.train(PARAMS, dtr, num_boost_round=300,
                    evals=[(dvl, "val")], early_stopping_rounds=20, verbose_eval=False)
    risk_ho = bst.predict(dho)
    c_ho = concordance_index(holdout["timeDiff"].values, -risk_ho, holdout["status"].values)
    return bst, risk_ho, c_ho, bst.best_iteration

INT_FEATURES = FEATURES + FUZZY_COLS + INTERACTION_COLS

results = []
t_start = time.time()
for i, task in enumerate(TASKS, 1):
    tid = task["task_id"]
    print(f"[{i:2d}/16] {tid}  loading", end=" ", flush=True)
    t0 = time.time()
    fit, holdout = load_task_with_fuzzy_and_interactions(task)
    print(f"({len(fit):,} fit, {len(holdout):,} holdout, {time.time()-t0:.0f}s)", flush=True)

    print(f"       fitting interactions ({len(INT_FEATURES)} features)", end=" ", flush=True)
    t0 = time.time()
    bst, risk_ho, c_ho, best_iter = fit_and_eval(fit, holdout, INT_FEATURES)
    dt = time.time() - t0
    print(f"({best_iter} rounds, hold-out C={c_ho:.4f}, {dt:.0f}s)", flush=True)

    pd.DataFrame({
        "risk": risk_ho,
        "timeDiff": holdout["timeDiff"].values,
        "status":   holdout["status"].values,
        "timestamp": holdout["timestamp"].values,
    }).to_csv(INT_DIR / f"{tid}_holdout_preds.csv", index=False)

    pa = pd.read_csv(AUG_DIR / f"{tid}_holdout_preds.csv")
    c_aug = concordance_index(pa["timeDiff"], -pa["risk"], pa["status"])
    results.append({
        "task_id": tid,
        "n_holdout": len(holdout),
        "c_int": float(c_ho),
        "c_aug": float(c_aug),
        "delta_int_vs_aug": float(c_ho - c_aug),
    })
    del fit, holdout, bst; gc.collect()

print(f"\ntotal time: {(time.time()-t_start)/60:.1f} min")
res_df = pd.DataFrame(results)
print()
print(res_df.round(4).to_string(index=False))
res_df.to_csv(INT_DIR / "interaction_results.csv", index=False)
print(f"\nmean delta vs aug: {res_df['delta_int_vs_aug'].mean():+.4f}")
print(f"wins vs aug: {(res_df['delta_int_vs_aug'] > 0).sum()}/{len(res_df)}")

[ 1/16] Borrow_Deposit  loading (784,167 fit, 98,041 holdout, 10s)
       fitting interactions (97 features) (51 rounds, hold-out C=0.7149, 15s)
[ 2/16] Borrow_Repay  loading (757,265 fit, 88,543 holdout, 9s)
       fitting interactions (97 features) (158 rounds, hold-out C=0.8241, 20s)
[ 3/16] Borrow_Withdraw  loading (785,318 fit, 98,106 holdout, 9s)
       fitting interactions (97 features) (140 rounds, hold-out C=0.7397, 20s)
[ 4/16] Borrow_Liquidated  loading (786,960 fit, 98,420 holdout, 9s)
       fitting interactions (97 features) (41 rounds, hold-out C=0.8117, 11s)
[ 5/16] Deposit_Borrow  loading (1,725,041 fit, 440,300 holdout, 24s)
       fitting interactions (97 features) (73 rounds, hold-out C=0.9346, 37s)
[ 6/16] Deposit_Repay  loading (1,722,401 fit, 440,439 holdout, 24s)
       fitting interactions (97 features) (296 rounds, hold-out C=0.9544, 71s)
[ 7/16] Deposit_Withdraw  loading (1,714,498 fit, 253,694 holdout, 22s)
       fitting interactions (97 features) (298 roun

In [15]:
N_BOOT = 100
BLOCK_SIZE = 10
MAX_BOOT_N = 3000

def block_bootstrap_delta_safe(pi, pa, n_boot=N_BOOT, block_size=BLOCK_SIZE,
                               max_n=MAX_BOOT_N, seed=42):
    """Each model evaluated on its OWN labels and risks within each bootstrap draw.
    Fixes the cross-file row alignment bug."""
    rng = np.random.default_rng(seed)
    n_full = min(len(pi), len(pa))
    n = min(n_full, max_n)
    deltas = []
    for _ in range(n_boot):
        idx = []
        while len(idx) < n:
            start = rng.integers(0, n_full)
            length = rng.geometric(1.0 / block_size)
            for k in range(length):
                idx.append((start + k) % n_full)
                if len(idx) >= n: break
        idx = np.array(idx[:n])
        yi, ei, ri = pi["timeDiff"].values[idx], pi["status"].values[idx], pi["risk"].values[idx]
        ya, ea, ra = pa["timeDiff"].values[idx], pa["status"].values[idx], pa["risk"].values[idx]
        if ei.sum() < 5 or ea.sum() < 5:
            continue
        try:
            ci = concordance_index(yi, -ri, ei)
            ca = concordance_index(ya, -ra, ea)
            deltas.append(ci - ca)
        except Exception:
            continue
    return np.array(deltas)

# diagnostic on first task
tid0 = TASKS[0]["task_id"]
pi0 = pd.read_csv(INT_DIR / f"{tid0}_holdout_preds.csv").sort_values("timestamp").reset_index(drop=True)
pa0 = pd.read_csv(AUG_DIR / f"{tid0}_holdout_preds.csv").sort_values("timestamp").reset_index(drop=True)
print(f"ALIGNMENT CHECK ({tid0}):")
print(f"  pi rows: {len(pi0)}, pa rows: {len(pa0)}")
if len(pi0) == len(pa0):
    print(f"  timestamps match: {(pi0['timestamp'].values == pa0['timestamp'].values).all()}")
    print(f"  timeDiff match:   {(pi0['timeDiff'].values == pa0['timeDiff'].values).all()}")
    print(f"  status match:     {(pi0['status'].values == pa0['status'].values).all()}")
print()

boot_rows = []
for i, task in enumerate(TASKS, 1):
    tid = task["task_id"]
    pi = pd.read_csv(INT_DIR / f"{tid}_holdout_preds.csv").sort_values("timestamp").reset_index(drop=True)
    pa = pd.read_csv(AUG_DIR / f"{tid}_holdout_preds.csv").sort_values("timestamp").reset_index(drop=True)
    print(f"[{i}/{len(TASKS)}] {tid}  bootstrapping...", flush=True)
    d = block_bootstrap_delta_safe(pi, pa)   # <-- NEW FUNCTION NAME
    if len(d) == 0:
        continue
    mean_d = float(d.mean())
    lo, hi = np.quantile(d, [0.025, 0.975])
    if lo > 0:   verdict = "WIN"
    elif hi < 0: verdict = "LOSE"
    else:        verdict = "null"
    print(f"       delta={mean_d:+.4f}  CI=[{lo:+.4f},{hi:+.4f}]  {verdict}", flush=True)
    boot_rows.append({"task_id": tid, "delta_mean": mean_d,
                      "ci_lo": lo, "ci_hi": hi, "verdict": verdict})

boot_df = pd.DataFrame(boot_rows)
print()
print(boot_df.round(4).to_string(index=False))

mean_delta = boot_df["delta_mean"].mean()
n_wins   = (boot_df["verdict"] == "WIN").sum()
n_losses = (boot_df["verdict"] == "LOSE").sum()
n_null   = (boot_df["verdict"] == "null").sum()

print(f"\nmean delta: {mean_delta:+.4f}")
print(f"wins: {n_wins}  losses: {n_losses}  null: {n_null}")

print(f"\npre-registered decision (threshold +/- 0.005):")
if n_wins > n_losses and mean_delta > 0.005:
    print("  POSITIVE: fuzzy interactions improve on augmented model")
elif n_losses > n_wins and mean_delta < -0.005:
    print("  NEGATIVE: fuzzy interactions hurt performance")
else:
    print("  NULL: fuzzy interactions do not meaningfully change performance")

boot_df.to_csv(INT_DIR / "interaction_bootstrap.csv", index=False)

ALIGNMENT CHECK (Borrow_Deposit):
  pi rows: 98041, pa rows: 98041
  timestamps match: True
  timeDiff match:   True
  status match:     True

[1/16] Borrow_Deposit  bootstrapping...
       delta=+0.0021  CI=[-0.0023,+0.0069]  null
[2/16] Borrow_Repay  bootstrapping...
       delta=+0.0008  CI=[-0.0017,+0.0029]  null
[3/16] Borrow_Withdraw  bootstrapping...
       delta=+0.0082  CI=[+0.0018,+0.0152]  WIN
[4/16] Borrow_Liquidated  bootstrapping...
       delta=+0.0084  CI=[-0.0516,+0.0605]  null
[5/16] Deposit_Borrow  bootstrapping...
       delta=+0.0179  CI=[+0.0088,+0.0281]  WIN
[6/16] Deposit_Repay  bootstrapping...
       delta=-0.0003  CI=[-0.0018,+0.0016]  null
[7/16] Deposit_Withdraw  bootstrapping...
       delta=-0.0022  CI=[-0.0038,-0.0005]  LOSE
[8/16] Deposit_Liquidated  bootstrapping...
       delta=-0.0037  CI=[-0.0133,+0.0029]  null
[9/16] Repay_Borrow  bootstrapping...
       delta=+0.0006  CI=[-0.0018,+0.0028]  null
[10/16] Repay_Deposit  bootstrapping...
       delta=

diagnostics cells-

In [6]:
# audit: check if userLiquidationCount is cumulative-as-of or total
for task in TASKS[:4]:
    tid = task["task_id"]
    train_path = RAW_ROOT / task["train_path"].replace("\\", "/")
    df = pd.read_csv(train_path).sort_values("timestamp").reset_index(drop=True)
    # for a sample user, check if userLiquidationCount is monotonically non-decreasing over time
    sample_user = df["user"].iloc[len(df)//2] if "user" in df.columns else None
    if sample_user:
        u = df[df["user"] == sample_user].sort_values("timestamp")
        if "userLiquidationCount" in u.columns and len(u) > 1:
            counts = u["userLiquidationCount"].values
            is_monotonic = np.all(np.diff(counts) >= 0)
            print(f"{tid}: user {sample_user}, n_rows={len(u)}, "
                  f"userLiqCount monotonic={is_monotonic}, values={counts[:5]}")

Borrow_Deposit: user 0xf94c5cb92c508eda709c17ec36c6c3167fff1a39, n_rows=4, userLiqCount monotonic=True, values=[0 0 0 0]
Borrow_Repay: user 0x00c6d212501fe3e3a8fc19871ff0ae46c342e5fd, n_rows=17, userLiqCount monotonic=True, values=[0 0 0 0 0]
Borrow_Withdraw: user 0x9f08dfb7dc3855c68fd44038998edc7c7fb4a651, n_rows=80, userLiqCount monotonic=True, values=[0 0 0 0 0]
Borrow_Liquidated: user 0xceb3e685565e50a06162f29387cd11da349e49b3, n_rows=20, userLiqCount monotonic=True, values=[0 0 0 0 0]


In [8]:
import os
# search for any train.csv files to see where the dataset actually lives
hits = []
for root, dirs, files in os.walk("."):
    if "train.csv" in files:
        hits.append(os.path.join(root, "train.csv"))
    if len(hits) >= 5:
        break
print(f"found {len(hits)} train.csv files")
for h in hits[:5]:
    print(f"  {h}")

# also check if features cache exists anywhere
feat_files = []
for root, dirs, files in os.walk("data"):
    for f in files:
        if f.endswith("_features.parquet"):
            feat_files.append(os.path.join(root, f))
            if len(feat_files) >= 3:
                break
    if len(feat_files) >= 3:
        break
print(f"\nfound {len(feat_files)} features.parquet files")
for f in feat_files:
    print(f"  {f}")

found 5 train.csv files
  .\data\raw\finsurvival\participant_data\participant_data\Borrow\Deposit\train.csv
  .\data\raw\finsurvival\participant_data\participant_data\Borrow\Liquidated\train.csv
  .\data\raw\finsurvival\participant_data\participant_data\Borrow\Repay\train.csv
  .\data\raw\finsurvival\participant_data\participant_data\Borrow\Withdraw\train.csv
  .\data\raw\finsurvival\participant_data\participant_data\Deposit\Borrow\train.csv

found 0 features.parquet files


In [10]:
import json
with open("data/processed/feature_list.json") as f:
    raw_FL = json.load(f)
print(f"top-level keys: {list(raw_FL.keys())}")
for k, v in raw_FL.items():
    if isinstance(v, list):
        print(f"  {k}: list of {len(v)}, first 3: {v[:3]}")
    elif isinstance(v, dict):
        print(f"  {k}: dict with keys {list(v.keys())[:5]}")
    else:
        print(f"  {k}: {type(v).__name__} = {v}")

top-level keys: ['n_features', 'features', 'user_feats', 'mkt_feats', 'time_feats', 'tx_feats', 'targets', 'metadata']
  n_features: int = 80
  features: list of 80, first 3: ['userBorrowSum', 'marketWithdrawAvgAmount', 'marketDepositAvgAmountUSD']
  user_feats: list of 31, first 3: ['userBorrowSum', 'userSecondsSinceFirstTransaction', 'userActiveDaysWeekly']
  mkt_feats: list of 25, first 3: ['marketWithdrawAvgAmount', 'marketDepositAvgAmountUSD', 'marketRepayAvgAmount']
  time_feats: list of 24, first 3: ['sinDayOfMonth', 'userSecondsSinceFirstTransaction', 'timeOfDay']
  tx_feats: list of 5, first 3: ['priceInUSD', 'amount', 'amountUSD']
  targets: list of 2, first 3: ['timeDiff', 'status']
  metadata: list of 8, first 3: ['user', 'pool', 'reserve']
